In [1]:
import json
import pandas as pd
from tqdm import tqdm

from graphs.legal_mcq_graph import run_mcq

In [2]:
QUESTIONS_PATH = r"F:\Thesis\project\403-vekalat\structured_questions.csv"
GOLD_PATH      = r"F:\Thesis\project\1-BaselineTest\GOLD\Gold.csv"

df_q = pd.read_csv(QUESTIONS_PATH)
df_gold = pd.read_csv(GOLD_PATH)

print("Questions:", len(df_q))
print("Gold:", len(df_gold))

display(df_q.head(2))
display(df_gold.head(2))

Questions: 119
Gold: 119


,question_number,category,question,options
0,1,حقوق مدنی,کدام یک از موارد زیر صحیح است؟,1) با توافق طرفین امکان از بین بردن سبب انفساخ...
1,2,حقوق مدنی,شخص «الف» حین رانندگی با خودرو سواری با شخص «ب...,1) دیه هر دو راننده صرفاً تا میزان دیه کامل و ...


,idx,Gold
0,1,3
1,2,2


In [3]:
# حذف سوال 89
if 89 in df_gold["idx"].values:
    df_gold = df_gold[df_gold["idx"] != 89].reset_index(drop=True)

multi_gold = {
    90: ["3", "4"],
    53: ["1", "2"],
    52: ["2", "4"],
    48: ["1", "3"],
    46: ["1", "3"],
    36: ["1", "4"],
    30: ["1", "2"],
    25: ["3", "4"],
    9:  ["1", "2"],
    4:  ["1", "4"],
    3:  ["2", "4"],
}

def normalize_gold_value(idx: int, gold_val: str) -> str:
    if idx in multi_gold:
        return ",".join(multi_gold[idx])

    val = str(gold_val).strip()
    digits = [ch for ch in val if ch.isdigit()]
    if not digits:
        return val
    if len(digits) == 1:
        return digits[0]
    return ",".join(digits)

df_gold["Gold"] = [
    normalize_gold_value(int(idx), gold)
    for idx, gold in zip(df_gold["idx"], df_gold["Gold"])
]

display(df_gold.head(5))

,idx,Gold
0,1,3
1,2,2
2,3,"2,4"
3,4,"1,4"
4,5,3


In [4]:
def is_correct(pred: str, gold: str) -> bool:
    gold_set = {g.strip() for g in str(gold).split(",") if g.strip()}
    return str(pred).strip() in gold_set

In [5]:
subset = df_q.copy()

# مثال: فقط چند سوال خاص
# subset = df_q[df_q["question_number"].isin([3,4,5,25,30,36,46,48,52,53,90])]

print("Subset size:", len(subset))


Subset size: 119


In [6]:
results_rows = []

for _, row in tqdm(subset.iterrows(), total=len(subset)):
    q_num = int(row["question_number"])
    category = str(row["category"]).strip()
    question = str(row["question"]).strip()
    options_raw = row["options"]

    gold_row = df_gold[df_gold["idx"] == q_num]
    gold_val = gold_row["Gold"].iloc[0] if not gold_row.empty else None

    # اجرای گراف از طریق helper
    out = run_mcq(
        question_number=q_num,
        category=category,
        question=question,
        options_raw=options_raw,
        max_revisions=2,
    )

    final_toon = out.get("final_toon") or {}
    pred_answer = final_toon.get("answer")
    confidence = final_toon.get("confidence")
    explanation = final_toon.get("explanation")

    critic_toon = out.get("critic_toon") or {}
    needs_revision = critic_toon.get("needs_revision")
    critic_issue = critic_toon.get("issue")
    critic_action = critic_toon.get("action")

    correct = None
    if gold_val is not None and pred_answer is not None:
        correct = is_correct(pred_answer, gold_val)

    results_rows.append({
        "question_number": q_num,
        "category": category,
        "gold": gold_val,
        "pred": pred_answer,
        "is_correct": correct,
        "confidence": confidence,
        "needs_revision": needs_revision,
        "critic_issue": critic_issue,
        "critic_action": critic_action,
        # برای خطایابی سریع:
        "context_preview": out.get("context_preview"),
        "docs_meta": out.get("docs_meta"),
    })


  0%|          | 0/119 [00:00<?, ?it/s]

📝 Query: کدام یک از موارد زیر صحیح است؟...
🔍 روش انتخاب شده: Semantic search
🔍 Semantic only: 20 نتیجه
   ✓ Retrieved: 20 documents
🔄 Reranking 20 سند (بدون law+article)...
   ✓ Reranked: top 5 documents


  1%|          | 1/119 [00:20<41:12, 20.95s/it]

📝 Query: شخص «الف» حین رانندگی با خودرو سواری با شخص «ب» (موتورسوار) تصادف می کند شخص «ال...
🎯 روش انتخاب شده: Metadata-aware
🎯 Metadata: law='قانون بیمه اجباری خسارات وارد شده به شخص ثالث در اثر حوادث ناشی از وسایل نقلیه' | None None
🔍 فیلتر فقط قانون...
   ✓ 0 سند (law_only)
🔍 Semantic fallback...
✅ نهایی: 20 نتیجه
   ✓ Retrieved: 20 documents
🔄 Reranking 20 سند (بدون law+article)...
   ✓ Reranked: top 5 documents


  2%|▏         | 2/119 [00:59<1:00:29, 31.02s/it]

...

 98%|█████████▊| 117/119 [38:58<00:26, 13.46s/it]

📝 Query: شورای اسلامی شهری اقدام به وضع مصوبه ای کرده و به موجب آن هرگونه استخدام در شورا...
🎯 روش انتخاب شده: Metadata-aware
🎯 Metadata: law='قانون مجازات اسلامی مصوب 1375' | None None
🔍 فیلتر فقط قانون...
   ✓ 10 سند (law_only)
🔍 Semantic fallback...
✅ نهایی: 24 نتیجه
   ✓ Retrieved: 24 documents
🔄 Reranking 24 سند (بدون law+article)...
   ✓ Reranked: top 6 documents


 99%|█████████▉| 118/119 [39:09<00:12, 12.65s/it]

📝 Query: در قانون اساسی به تعیین تکلیف کدام یک از امور مربوط به هیئت منصفه در جرایم سیاسی...
🎯 روش انتخاب شده: Metadata-aware
🎯 Metadata: law='قانون اساسی' | None None
🔍 فیلتر فقط قانون...
   ✓ 10 سند (law_only)
🔍 Semantic fallback...
✅ نهایی: 24 نتیجه
   ✓ Retrieved: 24 documents
🔄 Reranking 24 سند (بدون law+article)...
   ✓ Reranked: top 6 documents


100%|██████████| 119/119 [39:19<00:00, 19.82s/it]


In [8]:
df_res = pd.DataFrame(results_rows)
display(df_res.head(10))

overall_acc = df_res["is_correct"].mean()
print("Overall accuracy:", overall_acc)

print("\nAccuracy by category:")
print(df_res.groupby("category")["is_correct"].mean())


,question_number,category,gold,pred,is_correct,confidence,needs_revision,critic_issue,critic_action,context_preview,docs_meta
0,1,حقوق مدنی,3,3,True,5,False,no issue,accept answer,[منبع 1] (قانون) - قانون مدنی\nشماره_ماده: 73\...,"[{'i': 1, 'law': None, 'article_number': 73, '..."
1,2,حقوق مدنی,2,2,True,5,True,incorrect option selected,select option 2 instead,[منبع 1] (دادنامه)\ntitle: مبلغ مازاد بر دیه د...,"[{'i': 1, 'law': None, 'article_number': None,..."
2,3,حقوق مدنی,"2,4",4,True,5,False,no issue,accept answer,[منبع 1] (قانون) - قانون مدنی\nشماره_ماده: 139...,"[{'i': 1, 'law': None, 'article_number': 139, ..."
3,4,حقوق مدنی,"1,4",4,True,5,False,no issue,accept answer,[منبع 1] (دادنامه) - قانون آیین دادرسی مدنی | ...,"[{'i': 1, 'law': None, 'article_number': None,..."
4,5,حقوق مدنی,3,3,True,5,False,no issue,accept answer,[منبع 1] (وحدت رویه) - قانون مدنی\ntitle: عدم ...,"[{'i': 1, 'law': None, 'article_number': None,..."
5,6,حقوق مدنی,3,4,False,5,True,incorrect answer choice,select correct option based on sources,[منبع 1] (قانون) - قانون مدنی\nشماره_ماده: 795...,"[{'i': 1, 'law': None, 'article_number': 795, ..."
6,7,حقوق مدنی,1,4,False,5,True,contradicts source 4,must align with trade law,[منبع 1] (قانون) - قانون امور حسبی\nشماره_ماده...,"[{'i': 1, 'law': None, 'article_number': 94, '..."
7,8,حقوق مدنی,4,1,False,5,True,conflict with sources on right to rescind,revise explanation to align with Source 3,[منبع 1] (دادنامه) - قانون مدنی\ntitle: رد درخ...,"[{'i': 1, 'law': None, 'article_number': None,..."
8,9,حقوق مدنی,"1,2",4,False,5,False,no issue,accept answer,[منبع 1] (دادنامه)\ntitle: مسئولیت دولت ناشی ا...,"[{'i': 1, 'law': None, 'article_number': None,..."
9,10,حقوق مدنی,3,3,True,5,False,no issue,accept answer,[منبع 1] (قانون) - قانون مدنی\nشماره_ماده: 118...,"[{'i': 1, 'law': None, 'article_number': 1188,..."


Overall accuracy: 0.5546218487394958

Accuracy by category:
category
آیین دادرسی مدنی     0.450000
آیین دادرسی کیفری    0.750000
حقوق اساسی           0.750000
حقوق تجارت           0.550000
حقوق جزا             0.315789
حقوق مدنی            0.500000
Name: is_correct, dtype: float64


In [18]:
import pandas as pd
wrong_answers = df_res[df_res["is_correct"] == False]

wrong_answers.to_csv('wrong_answers.csv')